## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
# Imports

from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel
from dotenv import load_dotenv
import asyncio
import os
from IPython.display import display, Markdown
from pprint import pprint
import requests
load_dotenv(override=True)

# Constants

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern.

Costs are here: https://platform.openai.com/docs/pricing#web-search

## We will be making 4 Agents:

1. Search Agent - searches online given a search term using an OpenAI hosted tool
2. Planner Agent - given a query from the user, come up with searches
3. Report Agent - make a report on results
4. Push Agent - send a notification to the user's phone with a summary

## Our First Agent: Search Agent

Given a Search term, search for it on the internet and summarize results.

In [2]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4.1-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [3]:
message = "What are the most popular and successful AI Agent frameworks in May 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

As of May 2025, several AI agent frameworks have gained prominence for their capabilities in developing sophisticated AI applications:

**LangChain**: A leading open-source framework for building applications powered by large language models (LLMs). It offers a modular architecture that simplifies complex workflows, including context management and multi-step tasks. LangChain integrates seamlessly with various LLMs, such as OpenAI and Hugging Face, and supports semantic search and API interactions. It's particularly suited for conversational AI, research tools, and document analysis. ([curotec.com](https://www.curotec.com/insights/top-ai-agent-frameworks/?utm_source=openai))

**LangGraph**: An extension of LangChain, LangGraph focuses on multi-agent systems capable of collaboration and adaptation. It provides tools for coordinating multiple agents, visual graph-based workflows, and advanced error-handling capabilities. LangGraph is ideal for applications like storytelling, multi-step chatbots, and strategic planning tools. ([curotec.com](https://www.curotec.com/insights/top-ai-agent-frameworks/?utm_source=openai))

**CrewAI**: Designed for orchestrating role-playing AI agents, CrewAI enables the creation of teams of specialized agents working together on complex tasks. It features a role-based agent architecture, dynamic task planning, and inter-agent communication protocols. CrewAI is well-suited for building collaborative AI systems that require diverse expertise and coordinated efforts. ([chatbase.co](https://www.chatbase.co/blog/ai-agent-frameworks?utm_source=openai))

These frameworks are at the forefront of AI agent development, offering robust tools for creating intelligent, autonomous systems across various domains. 

Take a look at the trace

https://platform.openai.com/traces

## Our Second Agent: Planner Agent

Given a query, come up with 5 ideas for web searches that could be run.

Use Structured Outputs as our way to ensure the Agent provides what we need.

In [4]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 5

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# We use Pydantic objects to describe the Schema of the output

class WebSearchItem(BaseModel):
    reason: str
    "Your reasoning for why this search is important to the query."

    query: str
    "The search term to use for the web search."


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem]
    """A list of web searches to perform to best answer the query."""

# We pass in the Pydantic object to ensure the output follows the schema

planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4.1-mini",
    output_type=WebSearchPlan,
)

In [5]:

message = "What are the most popular and successful AI Agent frameworks in May 2025"

with trace("Search - Hareesh"):
    result = await Runner.run(planner_agent, message)
    pprint(result.final_output)

WebSearchPlan(searches=[WebSearchItem(reason='Identify the current top AI agent frameworks as of May 2025 based on popularity and success metrics.', query='most popular AI agent frameworks May 2025'), WebSearchItem(reason='Check recent expert reviews and industry analysis on AI agent frameworks in 2025.', query='best AI agent frameworks expert reviews 2025'), WebSearchItem(reason='Explore developer community preferences and usage statistics for AI agent frameworks in 2025.', query='AI agent frameworks usage statistics developer 2025'), WebSearchItem(reason='Find recent benchmark comparisons of AI agent frameworks performance in 2025.', query='AI agent frameworks performance benchmarks 2025'), WebSearchItem(reason='Look for news articles and reports on emerging successful AI agent frameworks in 2025.', query='emerging successful AI agent frameworks May 2025')])


In [6]:
for search in result.final_output.searches:
    print(search.query)
    print(search.reason)
    print("-"*100)

most popular AI agent frameworks May 2025
Identify the current top AI agent frameworks as of May 2025 based on popularity and success metrics.
----------------------------------------------------------------------------------------------------
best AI agent frameworks expert reviews 2025
Check recent expert reviews and industry analysis on AI agent frameworks in 2025.
----------------------------------------------------------------------------------------------------
AI agent frameworks usage statistics developer 2025
Explore developer community preferences and usage statistics for AI agent frameworks in 2025.
----------------------------------------------------------------------------------------------------
AI agent frameworks performance benchmarks 2025
Find recent benchmark comparisons of AI agent frameworks performance in 2025.
----------------------------------------------------------------------------------------------------
emerging successful AI agent frameworks May 2025
Look 

## Our Third Agent: Writer Agent

Take the results of internet searches and make a report

In [7]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str
    """A short 2-3 sentence summary of the findings."""

    markdown_report: str
    """The final report"""

    follow_up_questions: list[str]
    """Suggested topics to research further"""


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

## Our Fourth Agent: push notification

Just to show how easy it is to make a tool!

I'm using a nifty product called PushOver - to set this up yourself, visit https://pushover.net

In [8]:
@function_tool
def push(message: str):
    """Send a push notification with this brief message"""
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)
    return {"status": "success"}

In [9]:
push

FunctionTool(name='push', description='Send a push notification with this brief message', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11167f2e0>, strict_json_schema=True)

In [10]:
INSTRUCTIONS = """You are a member of a research team and will be provided with a short summary of a report.
When you receive the report summary, you send a push notification to the user using your tool, informing them that research is complete,
and including the report summary you receive"""


push_agent = Agent(
    name="Push agent",
    instructions=INSTRUCTIONS,
    tools=[push],
    model="gpt-4.1-mini",
    model_settings=ModelSettings(tool_choice="required")
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [11]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and send a push notification

In [12]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_push(report: ReportData):
    """ Use the push agent to send a notification to the user """
    print("Pushing...")
    result = await Runner.run(push_agent, report.short_summary)
    print("Push sent")
    return report

### Showtime!

In [13]:
query ="What are the most popular and successful AI Agent frameworks in May 2025"

with trace("Research trace - Hareesh"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_push(report)  
    print("Hooray!")
display(Markdown(report.markdown_report))

Starting research...
Planning searches...
Will perform 5 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Pushing...
Push sent
Hooray!


# Overview of AI Agent Frameworks in May 2025

As of May 2025, the field of Artificial Intelligence (AI) has experienced significant advancements, particularly in the development of AI agent frameworks. These frameworks serve as foundational tools for building sophisticated AI agents capable of executing complex tasks, interacting with users, and integrating seamlessly within various applications. This report explores the most popular and successful AI agent frameworks available in May 2025, outlining their unique features, architectures, applications, and considerations for developers when choosing an appropriate framework.

## 1. Introduction

In recent years, AI technology has evolved rapidly, leading to the emergence of diverse frameworks designed to facilitate the development of AI agents. Each framework provides varying levels of complexity, flexibility, and specialized features tailored to specific use cases, such as conversational AI, automated document processing, and multi-agent systems. Understanding these frameworks is crucial for developers, businesses, and researchers keen on leveraging AI capabilities.

### 1.1 Objective of the Report

The primary objective of this report is to evaluate and summarize key AI agent frameworks that have gained popularity and success in the industry as of May 2025. It aims to provide insights into each framework's capabilities, usability, and specific applications, guiding stakeholders in their choices when developing AI solutions.

## 2. Leading AI Agent Frameworks in 2025

### 2.1 LangChain

**Overview**: LangChain has become a leading open-source framework renowned for its ability to build applications powered by large language models (LLMs). As of mid-2024, it had amassed over 86,000 stars on GitHub, illustrating its wide adoption among developers.

**Key Features**:
- **Modular Architecture**: Enables developers to create complex and customizable AI workflows, streamlining the integration of external APIs and tools.
- **Integration Capabilities**: Supports various LLMs, including those from OpenAI and Hugging Face, allowing for comprehensive solutions in conversational agents and document automation.

**Limitations**: While powerful, LangChain can be challenging for novices due to its complexity and the requirement for careful setup and tutorials.

**Applications**: Particularly suited for applications in conversational AI, research tools, and automated document analysis.

### 2.2 LangGraph

**Overview**: Derived from LangChain, LangGraph focuses on multi-agent systems, providing an innovative graph-based structure to facilitate coordination among multiple agents.

**Key Features**:
- **Visual Workflows**: Offers visual graph-based workflows that enhance user understanding and management of agent interactions.
- **Coordination Tools**: Advanced error handling and interaction strategies improve the efficiency of multi-agent collaborations.

**Limitations**: Although powerful, it has a steeper learning curve compared to LangChain.

**Applications**: Ideal for storytelling, strategic planning tools, and any applications requiring intricate workflows and dynamic agent decision-making.

### 2.3 CrewAI

**Overview**: CrewAI is designed for orchestrating role-playing AI agents, promoting collaboration among agents that perform distinct roles.

**Key Features**:
- **Role-Based Architecture**: Facilitates the development of AI teams, allowing agents to communicate and coordinate tasks effectively.
- **User-Friendly**: Emphasizes ease of use, enabling developers to build AI systems quickly without extensive coding.

**Limitations**: While it simplifies the development process, some advanced features of competing frameworks may be absent.

**Applications**: Well-suited for collaborative AI development, particularly in projects that require diverse expertise.

### 2.4 AutoGen

**Overview**: Created by Microsoft Research, AutoGen is a robust framework that supports both autonomous and human-in-the-loop workflows, crucial for enterprise-level applications.

**Key Features**:
- **Multi-Agent Interaction**: Supports synchronous and asynchronous interactions among agents.
- **Flexibility and Complexity**: Allows for detailed algorithmic prompts, essential for many advanced applications.

**Limitations**: It may require significant technical expertise to implement effectively, especially for complex systems.

**Applications**: Ideal for enterprise applications requiring sophisticated multi-agent conversations and decision-making processes.

### 2.5 OpenAI Agents SDK

**Overview**: The OpenAI Agents SDK provides a structured environment for building AI agents that can reason, plan, and integrate with external APIs.

**Key Features**:
- **Specialized Toolset**: Offers an agent runtime and straightforward API for assigning roles and tools to agents.
- **Integration with OpenAI Models**: Easily connects with OpenAI's model endpoints, enhancing the capabilities of the agents developed with it.

**Limitations**: Still evolving, the SDK may not yet cover all use cases comprehensively.

**Applications**: Particularly useful for orchestrating complex, multi-step interactions, making it ideal for applications requiring robust planning and reasoning capabilities.

## 3. Emerging Frameworks

In parallel to the established frameworks, several emerging platforms have shown great promise:

### 3.1 AutoAgent

**Description**: A fully automated, zero-code framework allowing the creation of LLM agents through natural language input. It includes innovative features like an autonomous agent operating system and a Self-Play Agent Customization module.

### 3.2 Eliza

**Description**: An open-source AI agent framework suitable for Web3 applications, enabling users to interact seamlessly with blockchain applications and manage data without extensive coding.

### 3.3 Autono

**Description**: An advanced framework designed for adaptive decision-making and multi-agent collaboration, introducing memory transfer systems for shared, dynamically updated contexts among agents.

## 4. Considerations for Selecting AI Agent Frameworks

When choosing the right AI agent framework, several factors must be considered:
- **Complexity of Tasks**: Assess the complexity of tasks that the agents will handle. Simpler tasks may suit frameworks like CrewAI, while more complex requirements may necessitate LangChain or AutoGen.
- **Integration Needs**: Evaluate the integrations required with databases, APIs, and other external services, which can significantly influence your choice.
- **Scalability and Performance**: Consider how well the framework can scale as your application grows, particularly for commercial use.
- **Technical Expertise**: Assess the level of technical expertise in your team, as more complex frameworks require a deeper understanding of AI and software development.

## 5. Conclusion

The landscape of AI agent frameworks as of May 2025 showcases a diverse set of tools tailored for varying applications. Frameworks like LangChain and CrewAI offer powerful capabilities for developing sophisticated AI agents, each with unique strengths and challenges. Developers need to assess their project requirements, team expertise, and the specific features of each framework to select the most suitable tool for their needs. Ultimately, these frameworks represent the evolution of AI technologies, progressively democratizing access to AI development and enhancing capabilities in numerous applications ranging from conversational agents to complex multi-agent systems. 

## 6. References

1. Curotec. (2025). Top AI Agent Frameworks: Insights and Analysis. Retrieved from [curotec.com](https://www.curotec.com/insights/top-ai-agent-frameworks/?utm_source=openai)
2. Analytics Vidhya. (2025). Best AI Agent Frameworks to Consider in 2025. Retrieved from [analyticsvidhya.com](https://www.analyticsvidhya.com/blog/2024/07/ai-agent-frameworks/?utm_source=openai)
3. OpenAI. (2025). AI Agents SDK Overview. Retrieved from [langfuse.com](https://langfuse.com/blog/2025-03-19-ai-agent-comparison?utm_source=openai)
4. Turing. (2025). Comparative Analysis of AI Agent Frameworks. Retrieved from [turing.com](https://www.turing.com/resources/ai-agent-frameworks?utm_source=openai)

---

### As always, take a look at the trace

https://platform.openai.com/traces